# cytozip `dataloader` demo

Window-by-window loading of single-cell cytosine methylation data (deep-learning DataLoader use case).

- reference: `~/Ref/hg38/hg38_with_chrL.allc.cz`
- CG index: `~/Ref/hg38/hg38_with_chrL.CGN.cz`
- single-cell cz directory: `~/Projects/test_cytozip/benchmark/cz/` (all 2000 cells)
- window size: `binsize = 50`

Environment: conda `m3c` (cytozip installed from source, with the compiled Cython accelerator).

> Note: `iter_windows` streams — it only decompresses the `.cz` blocks covering each window, so peak memory is bounded by the read-segment (`~n_cells * max_sites`), not the whole chromosome. Use `load_chrom` / `load_chrom_matrix` only when you explicitly want the full `(n_cells, n_sites)` matrix in RAM (for chr1 ~4.75M CG sites × 2000 cells at `uint8` that is ~19 GB for mc + cov).

In [1]:
import os, glob, time
import numpy as np

import cytozip
from cytozip.dataloader import CzWindowLoader, resolve_cell_ids
print("cytozip:", cytozip.__file__)

REFERENCE = os.path.expanduser("~/Ref/hg38/hg38_with_chrL.allc.cz")
CGN_INDEX = os.path.expanduser("~/Ref/hg38/hg38_with_chrL.CGN.cz")
CELLS_DIR = os.path.expanduser("~/Projects/test_cytozip/benchmark/cz")

all_cells = sorted(glob.glob(os.path.join(CELLS_DIR, "*.cz")))
print(f"total single cells in directory: {len(all_cells)}")

# Use ALL cells; uint8 (mc/cov <= 255) keeps the per-window matrices small.
# Streaming iter_windows never materialises the whole chromosome, so 2000
# cells is fine memory-wise.
cells = all_cells
DTYPE = np.uint8
print(f"using {len(cells)} cells, dtype={np.dtype(DTYPE).name}")

cytozip: /home/x-wding2/Software/conda/m3c/lib/python3.10/site-packages/cytozip/__init__.py
total single cells in directory: 2000
using 2000 cells, dtype=uint8


## 1. Build the loader (opens reference / index / all cells once)

In [2]:
loader = CzWindowLoader(reference=REFERENCE, cells=cells, index=CGN_INDEX,
                        dtype=DTYPE, threads=16)

print("num chromosomes:", len(loader.chroms))
print("first 5 chromosomes:", loader.chroms[:5])
print("num cells:", len(loader.cell_ids))
print("first 3 cell_ids:", loader.cell_ids[:3])

num chromosomes: 26
first 5 chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13']
num cells: 2000
first 3 cell_ids: ['UWA7648_CX1819_NAC_1_P1-1-I3-A1', 'UWA7648_CX1819_NAC_1_P1-1-I3-A10', 'UWA7648_CX1819_NAC_1_P1-1-I3-A11']


## 2. Iterate chr1 window by window (CG sites only)

Each `Window` holds `chrom, start, end, pos, mc, cov`; `mc`/`cov` have shape `(n_cells, n_sites_in_window)`, with row order = `loader.cell_ids`.

In [3]:
CHROM = "chr1"
BINSIZE = 50

for i, w in enumerate(loader.iter_windows(CHROM, binsize=BINSIZE)):
    print(f"bin {i}: {w.chrom}:{w.start:,}-{w.end:,}  mc{w.mc.shape} cov{w.cov.shape}  sites={w.pos.shape[0]}")
    if i >= 4:
        break

bin 0: chr1:10,450-10,500  mc(2000, 12) cov(2000, 12)  sites=12
bin 1: chr1:10,500-10,550  mc(2000, 4) cov(2000, 4)  sites=4
bin 2: chr1:10,550-10,600  mc(2000, 10) cov(2000, 10)  sites=10
bin 3: chr1:10,600-10,650  mc(2000, 18) cov(2000, 18)  sites=18
bin 4: chr1:10,650-10,700  mc(2000, 25) cov(2000, 25)  sites=25


## 3. Time to fetch the first 10 bins

`iter_windows` streams: it decompresses only the `.cz` blocks covering each window's row-segment, so no whole-chromosome matrix is ever built.

In [4]:
# prefetch=True reads the next segment on a background thread while these
# windows are consumed (overlaps decompression with compute).
t0 = time.perf_counter()
bins = []
for w in loader.iter_windows(CHROM, binsize=BINSIZE, prefetch=True):
    bins.append(w)
    if len(bins) >= 10:
        break
elapsed_ms = (time.perf_counter() - t0) * 1e3
print(f"time to fetch first {len(bins)} bins: {elapsed_ms:.1f} ms")
print(f"bin0: {bins[0].mc.shape[0]} cells x {bins[0].mc.shape[1]} sites")

time to fetch first 10 bins: 635.5 ms
bin0: 2000 cells x 12 sites


In [5]:
# bins[:3]

## 4. Compute per-cell methylation level (mCG) for one window

In [6]:
w = bins[0]
cov = w.cov.astype(np.float32)
frac = np.where(cov > 0, w.mc / np.clip(cov, 1, None), np.nan)
per_cell = np.nanmean(frac, axis=1)  # mean mCG per cell in this window (ignoring uncovered sites)

print(f"window {w.chrom}:{w.start:,}-{w.end:,}  shape={frac.shape}")
for cid, m in zip(loader.cell_ids[:5], per_cell[:5]):
    print(f"{cid}: mCG={m:.3f}")

window chr1:10,450-10,500  shape=(2000, 12)
UWA7648_CX1819_NAC_1_P1-1-I3-A1: mCG=0.667
UWA7648_CX1819_NAC_1_P1-1-I3-A10: mCG=nan
UWA7648_CX1819_NAC_1_P1-1-I3-A11: mCG=nan
UWA7648_CX1819_NAC_1_P1-1-I3-A12: mCG=nan
UWA7648_CX1819_NAC_1_P1-1-I3-A14: mCG=nan


/tmp/ipykernel_2873754/1592952354.py:4: RuntimeWarning: Mean of empty slice
  per_cell = np.nanmean(frac, axis=1)  # mean mCG per cell in this window (ignoring uncovered sites)


## 5. Single-window random access via `load_region`

In [7]:
# Load a single genomic window [start, end) across all cells, reading only the
# .cz blocks covering it (low memory — never the whole chromosome).
pos_w, mc_w, cov_w = loader.load_region(CHROM, 1_000_000, 1_100_000)
print(f"{CHROM}:1,000,000-1,100,000  CG sites: {pos_w.shape[0]:,}"
      f"  matrix: {mc_w.shape} {mc_w.dtype}  ({len(loader.cell_ids)} cells)")
# NOTE: loader.load_chrom(CHROM) would build the full (n_cells, n_sites) matrix
# — for chr1 (~4.75M CG x 2000 cells at uint8) that is ~19 GB, so prefer
# load_region / iter_windows for large cohorts.

chr1:1,000,000-1,100,000  CG sites: 8,826  matrix: (2000, 8826) uint8  (2000 cells)


## 6. (Optional) Use as a PyTorch `IterableDataset`

In [8]:
try:
    import torch
    from cytozip.dataloader import CzWindowDataset
    # small subset just to show tensor output
    ds = CzWindowDataset(reference=REFERENCE, cells=all_cells[:50], chrom=CHROM,
                         binsize=BINSIZE, index=CGN_INDEX, dtype=DTYPE, threads=16,
                         to_torch=True)
    for w in ds:
        print("torch tensor:", type(w.mc).__name__, tuple(w.mc.shape), w.mc.dtype)
        break
    ds.close()
except ImportError:
    print("torch not installed, skipping")

torch tensor: Tensor (50, 12) torch.uint8


## 7. Close the loader

In [9]:
loader.close()
print("done")

done
